In [ ]:
# This code breaks up the larger tiff image file into smaller tiles for use by YOLO
# Note: We later didn't need YOLO because Roboflow was used for building detection and
# labelling.


In [1]:
import os
import rasterio
from rasterio.windows import Window

def tile_geotiff(image_path, output_dir, tile_size=640, overlap=0):
    """
    Slices a large GeoTIFF image into smaller square tiles for YOLO training.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    with rasterio.open(image_path) as src:
        width = src.width
        height = src.height
        
        # Calculate steps considering any overlap if needed
        step = tile_size - overlap
        tile_count = 0
        
        # Loop through rows (Y axis) and columns (X axis)
        for y in range(0, height, step):
            for x in range(0, width, step):
                
                # Handle edge tiles so they don't overshoot the image boundaries
                w_width = min(tile_size, width - x)
                w_height = min(tile_size, height - y)
                
                # Define the cropping window
                window = Window(x, y, w_width, w_height)
                
                # Read the pixel data inside this window
                # (Reading all image bands, e.g., RGB)
                data = src.read(window=window)
                
                # Skip mostly empty tiles on image borders
                if data.sum() == 0:
                    continue
                    
                # Update the geospatial metadata tags for the mini-tile
                meta = src.meta.copy()
                meta.update({
                    "height": w_height,
                    "width": w_width,
                    "transform": rasterio.windows.transform(window, src.transform)
                })
                
                # Save the new mini-tile image patch
                tile_filename = f"tile_y{y}_x{x}.tif"
                tile_filepath = os.path.join(output_dir, tile_filename)
                
                with rasterio.open(tile_filepath, "w", **meta) as dest:
                    dest.write(data)
                
                tile_count += 1
                
        print(f"Successfully generated {tile_count} tiles of size {tile_size}x{tile_size} inside '{output_dir}'.")

# Run the slicer
tile_geotiff(
    image_path="C:/Users/user/Downloads/python_projects/dn_315_supply_image_clipped.tif", 
    output_dir="C:/Users/user/Downloads/python_projects/yolo_dataset/images", 
    tile_size=640,
    overlap=64 # Optional: 10% overlap helps catch buildings cut off at tile borders
)


Successfully generated 133 tiles of size 640x640 inside 'C:/Users/user/Downloads/python_projects/yolo_dataset/images'.
